# Main panel Granger regression: hourly AI risk and stock returns

This notebook runs the **pre-specified main regression only**:

- Outcome: `log_ret_oc` for the 20 stocks during the six New York windows from 10:00–11:00 through 15:00–16:00.
- Main predictor: `AI_Risk_total` from the hourly X News risk index.
- Return lag order $q$: selected once from 1–3 by BIC on a common return-only panel sample.
- Risk lag order $p$: estimated separately for 1, 2 and 3 hours.
- Controls: ticker fixed effects, target-hour fixed effects and weekday fixed effects.
- Inference: conventional nested F test plus a joint Wald test using two-way clustered covariance (ticker and exact target timestamp).

An index row labelled `hour = h` summarises news during $[h,h+1)$. It is only available at $h+1$. Therefore a return during $[t,t+1)$ uses risk from $[t-1,t)$ or earlier; contemporaneous risk is never used.

The term *Granger causality* here means incremental predictive content, not structural or philosophical causality. The X collection hit its query result caps, so all results describe the **observed X News sample**, not complete X-platform coverage.

In [1]:
from pathlib import Path
import platform
import sys
import warnings

import numpy as np
import pandas as pd
import scipy
import statsmodels
import statsmodels.api as sm
from IPython.display import Markdown, display
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.6g}")

ROOT = Path.cwd().resolve()
if not (ROOT / "download_hourly_returns.py").exists():
    raise RuntimeError("Run this notebook from the project root directory.")

REGULAR_PATH = ROOT / "data" / "hourly_10_16" / "hourly_returns_long.csv"
BRIDGE_PATH = ROOT / "data" / "premarket_bridge_07_10" / "hourly_returns_long.csv"
RISK_PATH = ROOT / "data" / "hourly_ai_risk_index" / "x" / "hourly_ai_risk_index.csv"

for path in (REGULAR_PATH, BRIDGE_PATH, RISK_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

environment = pd.DataFrame(
    {
        "component": ["Python", "pandas", "NumPy", "SciPy", "statsmodels", "platform"],
        "version": [
            platform.python_version(),
            pd.__version__,
            np.__version__,
            scipy.__version__,
            statsmodels.__version__,
            platform.platform(),
        ],
    }
)
display(environment)
print("Interpreter:", sys.executable)
print("Project root:", ROOT)

,component,version
0,Python,3.14.7
1,pandas,3.0.6
2,NumPy,2.5.3
3,SciPy,1.18.1
4,statsmodels,0.15.0
5,platform,Windows-11-10.0.26200-SP0


Interpreter: C:\Users\Jerry Cai\Desktop\NLP_Assignment4_AI_risk_and_AI_development\.venv\Scripts\python.exe
Project root: C:\Users\Jerry Cai\Desktop\NLP_Assignment4_AI_risk_and_AI_development


## 1. Load and validate the three hourly datasets

The bridge file is used only to supply lagged returns before 10:00. It is never used as the dependent variable. Missing bridge bars and unknown risk hours remain missing.

In [2]:
def parse_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().eq("true")


def load_return_file(path: Path, expected_mode: str) -> pd.DataFrame:
    frame = pd.read_csv(path)
    required = {
        "ticker", "trading_date", "bar_start_utc", "bar_end_utc",
        "duration_minutes", "window_complete", "ohlc_valid", "nonpositive_price",
        "split_event", "log_ret_oc", "return_requires_review",
    }
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    frame["bar_start_utc"] = pd.to_datetime(frame["bar_start_utc"], utc=True)
    frame["bar_end_utc"] = pd.to_datetime(frame["bar_end_utc"], utc=True)
    frame["window_complete"] = parse_bool(frame["window_complete"])
    frame["ohlc_valid"] = parse_bool(frame["ohlc_valid"])
    frame["nonpositive_price"] = parse_bool(frame["nonpositive_price"])
    frame["split_event"] = parse_bool(frame["split_event"])
    frame["return_requires_review"] = parse_bool(frame["return_requires_review"])
    frame["duration_minutes"] = pd.to_numeric(frame["duration_minutes"], errors="coerce")
    frame["log_ret_oc"] = pd.to_numeric(frame["log_ret_oc"], errors="coerce")
    if "window_mode" in frame.columns and set(frame["window_mode"].dropna().unique()) != {expected_mode}:
        raise ValueError(f"Unexpected window mode in {path}")
    if frame.duplicated(["ticker", "bar_start_utc"]).any():
        raise ValueError(f"Duplicate ticker/start timestamps in {path}")
    return frame


regular = load_return_file(REGULAR_PATH, "clock_10_16")
bridge = load_return_file(BRIDGE_PATH, "bridge_07_10")
risk = pd.read_csv(RISK_PATH)

risk_required = {
    "hour", "AI_Risk_total", "coverage_status", "coverage_complete",
    "is_backtest_eligible_observed_sample", "is_backtest_eligible_strict",
}
missing = risk_required.difference(risk.columns)
if missing:
    raise ValueError(f"Risk index is missing columns: {sorted(missing)}")

risk["hour"] = pd.to_datetime(risk["hour"], utc=True)
risk["AI_Risk_total"] = pd.to_numeric(risk["AI_Risk_total"], errors="coerce")
risk["observed_eligible"] = parse_bool(risk["is_backtest_eligible_observed_sample"])
risk["strict_eligible"] = parse_bool(risk["is_backtest_eligible_strict"])
if risk["hour"].duplicated().any():
    raise ValueError("Risk index contains duplicate hourly timestamps")
if len(risk) > 1 and not risk["hour"].sort_values().diff().dropna().eq(pd.Timedelta(hours=1)).all():
    raise ValueError("Risk index is not a continuous hourly grid")

regular_hours = set(regular["bar_start_utc"].dt.tz_convert("America/New_York").dt.hour.unique())
bridge_hours = set(bridge["bar_start_utc"].dt.tz_convert("America/New_York").dt.hour.unique())
assert regular_hours == set(range(10, 16)), regular_hours
assert bridge_hours == {7, 8, 9}, bridge_hours
assert regular["duration_minutes"].eq(60).all()

targets = regular.loc[
    regular["window_complete"]
    & regular["ohlc_valid"]
    & ~regular["nonpositive_price"]
    & ~regular["split_event"]
    & ~regular["return_requires_review"]
    & regular["log_ret_oc"].notna()
].copy()
targets = targets.rename(columns={"log_ret_oc": "target_return"})
targets["target_start_ny"] = targets["bar_start_utc"].dt.tz_convert("America/New_York")
targets["target_end_ny"] = targets["bar_end_utc"].dt.tz_convert("America/New_York")
targets["target_hour"] = targets["target_start_ny"].dt.hour.astype(str).str.zfill(2) + ":00"
targets["trading_date"] = targets["trading_date"].astype(str)
targets["weekday"] = pd.to_datetime(targets["trading_date"]).dt.day_name()

return_grid = pd.concat(
    [
        bridge[["ticker", "trading_date", "bar_start_utc", "window_complete", "ohlc_valid", "nonpositive_price", "split_event", "return_requires_review", "log_ret_oc"]],
        regular[["ticker", "trading_date", "bar_start_utc", "window_complete", "ohlc_valid", "nonpositive_price", "split_event", "return_requires_review", "log_ret_oc"]],
    ],
    ignore_index=True,
)
usable_return = (
    return_grid["window_complete"]
    & return_grid["ohlc_valid"]
    & ~return_grid["nonpositive_price"]
    & ~return_grid["split_event"]
    & ~return_grid["return_requires_review"]
)
return_grid.loc[~usable_return, "log_ret_oc"] = np.nan
return_grid["trading_date"] = return_grid["trading_date"].astype(str)
if return_grid.duplicated(["ticker", "bar_start_utc"]).any():
    raise ValueError("Combined bridge/regular return grid has duplicate keys")

return_lookup = return_grid.set_index(["ticker", "bar_start_utc"])["log_ret_oc"]
date_lookup = return_grid.set_index(["ticker", "bar_start_utc"])["trading_date"]
risk_value_lookup = risk.set_index("hour")["AI_Risk_total"]
risk_eligible_lookup = risk.set_index("hour")["observed_eligible"]

for lag in range(1, 4):
    return_key_time = targets["bar_start_utc"] - pd.Timedelta(hours=lag)
    return_keys = pd.MultiIndex.from_arrays([targets["ticker"], return_key_time])
    targets[f"return_lag_{lag}"] = return_lookup.reindex(return_keys).to_numpy()
    lag_dates = date_lookup.reindex(return_keys).to_numpy()
    same_day = lag_dates == targets["trading_date"].to_numpy()
    targets.loc[~same_day, f"return_lag_{lag}"] = np.nan

    risk_hour = targets["bar_start_utc"] - pd.Timedelta(hours=lag)
    targets[f"risk_hour_lag_{lag}"] = risk_hour
    targets[f"risk_lag_{lag}"] = risk_value_lookup.reindex(risk_hour).to_numpy()
    eligible = risk_eligible_lookup.reindex(risk_hour)
    targets[f"risk_eligible_lag_{lag}"] = eligible.fillna(False).astype(bool).to_numpy()

targets = targets.sort_values(["bar_start_utc", "ticker"]).reset_index(drop=True)

assert (targets["risk_hour_lag_1"] + pd.Timedelta(hours=1) == targets["bar_start_utc"]).all()
assert not targets.duplicated(["ticker", "bar_start_utc"]).any()

audit = pd.DataFrame(
    [
        ("regular target rows", len(targets)),
        ("tickers", targets["ticker"].nunique()),
        ("regular trading dates", targets["trading_date"].nunique()),
        ("bridge expected rows", len(bridge)),
        ("bridge complete rows", int(bridge["window_complete"].sum())),
        ("risk hourly rows", len(risk)),
        ("risk observed-sample eligible rows", int(risk["observed_eligible"].sum())),
        ("risk strict-coverage eligible rows", int(risk["strict_eligible"].sum())),
        ("risk queries at result cap", int(pd.to_numeric(risk.get("queries_at_result_cap"), errors="coerce").max())),
    ],
    columns=["check", "value"],
)
display(audit)

example_row = targets.loc[
    targets["ticker"].eq("MSFT")
    & targets["target_start_ny"].dt.hour.eq(10)
    & targets["risk_lag_3"].notna()
].iloc[0]
alignment_example = pd.DataFrame(
    {
        "role": ["target return", "lag 1 risk/return", "lag 2 risk/return", "lag 3 risk/return"],
        "window_start_NY": [
            example_row["target_start_ny"],
            example_row["target_start_ny"] - pd.Timedelta(hours=1),
            example_row["target_start_ny"] - pd.Timedelta(hours=2),
            example_row["target_start_ny"] - pd.Timedelta(hours=3),
        ],
        "window_end_NY": [
            example_row["target_end_ny"],
            example_row["target_start_ny"],
            example_row["target_start_ny"] - pd.Timedelta(hours=1),
            example_row["target_start_ny"] - pd.Timedelta(hours=2),
        ],
    }
)
display(Markdown("**Exact no-look-ahead example (MSFT):**"))
display(alignment_example)

,check,value
0,regular target rows,2760
1,tickers,20
2,regular trading dates,23
3,bridge expected rows,1380
4,bridge complete rows,887
5,risk hourly rows,723
6,risk observed-sample eligible rows,622
7,risk strict-coverage eligible rows,0
8,risk queries at result cap,33


**Exact no-look-ahead example (MSFT):**

,role,window_start_NY,window_end_NY
0,target return,2026-09-02 10:00:00-04:00,2026-09-02 11:00:00-04:00
1,lag 1 risk/return,2026-09-02 09:00:00-04:00,2026-09-02 10:00:00-04:00
2,lag 2 risk/return,2026-09-02 08:00:00-04:00,2026-09-02 09:00:00-04:00
3,lag 3 risk/return,2026-09-02 07:00:00-04:00,2026-09-02 08:00:00-04:00


## 2. Select the return lag order $q$ by BIC

The BIC comparison uses exactly the same observations for $q=1,2,3$: rows for which all three return lags exist within the same trading day. Risk values are not used in this selection. Lower BIC is preferred; ties within numerical precision are resolved in favour of the smaller lag order.

In [3]:
def fixed_effects(frame: pd.DataFrame) -> pd.DataFrame:
    categories = frame[["ticker", "target_hour", "weekday"]].astype(str)
    return pd.get_dummies(
        categories,
        columns=["ticker", "target_hour", "weekday"],
        prefix=["ticker", "hour", "weekday"],
        drop_first=True,
        dtype=float,
    )


def design_matrix(frame: pd.DataFrame, q: int, p: int = 0) -> pd.DataFrame:
    matrix = pd.DataFrame({"const": np.ones(len(frame), dtype=float)}, index=frame.index)
    for lag in range(1, q + 1):
        matrix[f"return_lag_{lag}"] = frame[f"return_lag_{lag}"].astype(float)
    for lag in range(1, p + 1):
        matrix[f"risk_lag_{lag}"] = frame[f"risk_lag_{lag}"].astype(float)
    matrix = pd.concat([matrix, fixed_effects(frame)], axis=1)
    return matrix.astype(float)


bic_mask = targets["target_return"].notna()
for lag in range(1, 4):
    bic_mask &= targets[f"return_lag_{lag}"].notna()
bic_sample = targets.loc[bic_mask].copy()

bic_rows = []
for q in (1, 2, 3):
    X = design_matrix(bic_sample, q=q, p=0)
    fit = sm.OLS(bic_sample["target_return"].astype(float), X, missing="raise").fit()
    bic_rows.append(
        {
            "q_return_lags": q,
            "n_obs_common_sample": int(fit.nobs),
            "n_parameters": int(fit.df_model + 1),
            "log_likelihood": float(fit.llf),
            "AIC": float(fit.aic),
            "BIC": float(fit.bic),
            "adjusted_R2": float(fit.rsquared_adj),
        }
    )

bic_results = pd.DataFrame(bic_rows).sort_values("q_return_lags").reset_index(drop=True)
Q_SELECTED = int(bic_results.loc[bic_results["BIC"].idxmin(), "q_return_lags"])
bic_results["delta_BIC"] = bic_results["BIC"] - bic_results["BIC"].min()
bic_results["selected"] = bic_results["q_return_lags"].eq(Q_SELECTED)

display(
    bic_results.style
    .format({"log_likelihood": "{:.3f}", "AIC": "{:.3f}", "BIC": "{:.3f}", "delta_BIC": "{:.3f}", "adjusted_R2": "{:.6f}"})
    .hide(axis="index")
)
display(Markdown(f"**Selected return lag order: $q={Q_SELECTED}$ (minimum BIC).**"))
near_ties = bic_results.loc[(bic_results["delta_BIC"] > 0) & (bic_results["delta_BIC"] < 2)]
if not near_ties.empty:
    alternatives = ", ".join(
        f"q={int(row.q_return_lags)} (ΔBIC={row.delta_BIC:.3f})"
        for row in near_ties.itertuples()
    )
    display(
        Markdown(
            f"**BIC提示：{alternatives} 与最优值差距小于2，拟合证据接近；"
            f"主回归按预先约定的简约原则保留 q={Q_SELECTED}，高阶收益滞后留作稳健性检验。**"
        )
    )

q_return_lags,n_obs_common_sample,n_parameters,log_likelihood,AIC,BIC,adjusted_R2,delta_BIC,selected
1,2031,30,7569.505,-15079.011,-14910.522,0.008588,0.000,True
2,2031,31,7570.033,-15078.065,-14903.961,0.008608,6.562,False
3,2031,32,7576.918,-15089.837,-14910.116,0.014815,0.407,False


**Selected return lag order: $q=1$ (minimum BIC).**

**BIC提示：q=3 (ΔBIC=0.407) 与最优值差距小于2，拟合证据接近；主回归按预先约定的简约原则保留 q=1，高阶收益滞后留作稳健性检验。**

## 3. Estimate the main panel regressions

For comparability, all three values $p\in\{1,2,3\}$ use one common sample that has valid risk lags through $p=3$. Within each value of $p$, the restricted and unrestricted regressions also use this identical sample.

Restricted model:

$$r_{i,t}=\alpha_i+\sum_{j=1}^{q}\phi_jr_{i,t-j}+\text{hour FE}+\text{weekday FE}+\epsilon_{i,t}$$

Unrestricted model:

$$r_{i,t}=\alpha_i+\sum_{j=1}^{q}\phi_jr_{i,t-j}+\sum_{j=1}^{p}\beta_jRisk_{t-j}+\text{hour FE}+\text{weekday FE}+\epsilon_{i,t}$$

The Granger null is $H_0:\beta_1=\cdots=\beta_p=0$. The clustered Wald p-value is the primary inference; the classical nested F p-value is shown for transparency.

In [4]:
RISK_SD = float(risk.loc[risk["observed_eligible"], "AI_Risk_total"].std(ddof=1))
if not np.isfinite(RISK_SD) or RISK_SD <= 0:
    raise ValueError("AI_Risk_total has no usable variation")


def common_main_sample(frame: pd.DataFrame, q: int, max_p: int = 3) -> pd.DataFrame:
    mask = frame["target_return"].notna()
    for lag in range(1, q + 1):
        mask &= frame[f"return_lag_{lag}"].notna()
    for lag in range(1, max_p + 1):
        mask &= frame[f"risk_lag_{lag}"].notna() & frame[f"risk_eligible_lag_{lag}"]
    return frame.loc[mask].copy()


def fit_main_model(frame: pd.DataFrame, q: int, p: int):
    sample = frame.copy()
    if sample.empty:
        raise ValueError(f"No complete observations for q={q}, p={p}")

    y = sample["target_return"].astype(float)
    X_restricted = design_matrix(sample, q=q, p=0)
    X_unrestricted = design_matrix(sample, q=q, p=p)
    if np.linalg.matrix_rank(X_unrestricted.to_numpy()) != X_unrestricted.shape[1]:
        raise ValueError(f"Design matrix is rank deficient for q={q}, p={p}")

    restricted = sm.OLS(y, X_restricted, missing="raise").fit()
    unrestricted = sm.OLS(y, X_unrestricted, missing="raise").fit()
    if restricted.nobs != unrestricted.nobs:
        raise AssertionError("Restricted and unrestricted models use different samples")

    ticker_codes = pd.Categorical(sample["ticker"]).codes
    time_codes = pd.Categorical(sample["bar_start_utc"]).codes
    groups = np.column_stack([ticker_codes, time_codes])
    clustered = unrestricted.get_robustcov_results(
        cov_type="cluster",
        groups=groups,
        use_correction=True,
        df_correction=True,
    )

    names = list(X_unrestricted.columns)
    risk_names = [f"risk_lag_{lag}" for lag in range(1, p + 1)]
    risk_positions = [names.index(name) for name in risk_names]
    restriction = np.zeros((p, len(names)), dtype=float)
    for row_number, position in enumerate(risk_positions):
        restriction[row_number, position] = 1.0

    wald_f = clustered.wald_test(restriction, use_f=True, scalar=True)
    wald_chi2 = clustered.wald_test(restriction, use_f=False, scalar=True)
    classical_f, classical_f_p, classical_df = unrestricted.compare_f_test(restricted)

    params = np.asarray(clustered.params, dtype=float)
    covariance = np.asarray(clustered.cov_params(), dtype=float)
    beta_sum_vector = np.zeros(len(names), dtype=float)
    beta_sum_vector[risk_positions] = 1.0
    beta_sum = float(beta_sum_vector @ params)
    beta_sum_variance = float(beta_sum_vector @ covariance @ beta_sum_vector)
    beta_sum_se = np.sqrt(beta_sum_variance) if beta_sum_variance >= 0 else np.nan
    inference_df = float(wald_f.df_denom)
    critical_value = float(stats.t.ppf(0.975, inference_df))
    beta_sum_low = beta_sum - critical_value * beta_sum_se if np.isfinite(beta_sum_se) else np.nan
    beta_sum_high = beta_sum + critical_value * beta_sum_se if np.isfinite(beta_sum_se) else np.nan

    summary = {
        "risk_lags_p": p,
        "return_lags_q": q,
        "n_obs": int(unrestricted.nobs),
        "n_tickers": int(sample["ticker"].nunique()),
        "n_target_timestamps": int(sample["bar_start_utc"].nunique()),
        "n_trading_days": int(sample["trading_date"].nunique()),
        "n_parameters": int(unrestricted.df_model + 1),
        "residual_df": int(unrestricted.df_resid),
        "R2": float(unrestricted.rsquared),
        "adjusted_R2": float(unrestricted.rsquared_adj),
        "risk_beta_sum": beta_sum,
        "risk_beta_sum_cluster_se": beta_sum_se,
        "effect_per_1SD_risk": beta_sum * RISK_SD,
        "effect_1SD_CI_low": beta_sum_low * RISK_SD,
        "effect_1SD_CI_high": beta_sum_high * RISK_SD,
        "classical_F": float(classical_f),
        "classical_F_df": int(classical_df),
        "classical_F_p": float(classical_f_p),
        "clustered_Wald_F": float(np.asarray(wald_f.statistic).squeeze()),
        "clustered_Wald_F_df_num": p,
        "clustered_Wald_F_df_denom": float(wald_f.df_denom),
        "clustered_Wald_F_p": float(np.asarray(wald_f.pvalue).squeeze()),
        "clustered_Wald_chi2": float(np.asarray(wald_chi2.statistic).squeeze()),
        "clustered_Wald_df": p,
        "clustered_Wald_chi2_p": float(np.asarray(wald_chi2.pvalue).squeeze()),
    }

    detail_rows = []
    for lag, position in enumerate(risk_positions, start=1):
        coefficient = float(params[position])
        variance = float(covariance[position, position])
        standard_error = np.sqrt(variance) if variance >= 0 else np.nan
        t_value = coefficient / standard_error if standard_error > 0 else np.nan
        p_value = 2 * stats.t.sf(abs(t_value), inference_df) if np.isfinite(t_value) else np.nan
        detail_rows.append(
            {
                "risk_lags_p": p,
                "term": f"AI_Risk_total_lag_{lag}",
                "coefficient": coefficient,
                "two_way_cluster_se": standard_error,
                "t": t_value,
                "inference_df": inference_df,
                "p_value": p_value,
                "effect_per_1SD_risk": coefficient * RISK_SD,
            }
        )
    return summary, detail_rows, sample


summary_rows = []
coefficient_rows = []
samples_by_p = {}
MAIN_COMMON_SAMPLE = common_main_sample(targets, q=Q_SELECTED, max_p=3)
for p in (1, 2, 3):
    summary, details, sample = fit_main_model(MAIN_COMMON_SAMPLE, q=Q_SELECTED, p=p)
    summary_rows.append(summary)
    coefficient_rows.extend(details)
    samples_by_p[p] = sample

main_results = pd.DataFrame(summary_rows).sort_values("risk_lags_p").reset_index(drop=True)
main_results["BH_q_value"] = multipletests(main_results["clustered_Wald_F_p"], method="fdr_bh")[1]
main_results["significant_BH_5pct"] = main_results["BH_q_value"] < 0.05
coefficient_results = pd.DataFrame(coefficient_rows)

main_display = main_results.assign(
    effect_1SD_percent=lambda data: 100 * np.expm1(data["effect_per_1SD_risk"]),
    effect_1SD_CI_low_percent=lambda data: 100 * np.expm1(data["effect_1SD_CI_low"]),
    effect_1SD_CI_high_percent=lambda data: 100 * np.expm1(data["effect_1SD_CI_high"]),
)[[
    "risk_lags_p", "return_lags_q", "n_obs", "n_target_timestamps",
    "risk_beta_sum", "effect_1SD_percent", "effect_1SD_CI_low_percent",
    "effect_1SD_CI_high_percent", "clustered_Wald_F", "clustered_Wald_F_df_num",
    "clustered_Wald_F_df_denom", "clustered_Wald_F_p", "BH_q_value",
    "significant_BH_5pct",
]].rename(
    columns={
        "risk_lags_p": "风险滞后p",
        "return_lags_q": "收益滞后q",
        "n_obs": "面板观测数",
        "n_target_timestamps": "独立目标小时数",
        "risk_beta_sum": "风险系数合计",
        "effect_1SD_percent": "风险+1SD的收益影响(%)",
        "effect_1SD_CI_low_percent": "95%CI下限(%)",
        "effect_1SD_CI_high_percent": "95%CI上限(%)",
        "clustered_Wald_F": "双向聚类联合F",
        "clustered_Wald_F_df_num": "分子df",
        "clustered_Wald_F_df_denom": "分母df",
        "clustered_Wald_F_p": "原始p值",
        "BH_q_value": "BH校正q值",
        "significant_BH_5pct": "FDR 5%显著",
    }
)
display(Markdown("### 主回归结果表"))
display(
    main_display.style.format(
        {
            "风险系数合计": "{:.8f}",
            "风险+1SD的收益影响(%)": "{:.5f}",
            "95%CI下限(%)": "{:.5f}",
            "95%CI上限(%)": "{:.5f}",
            "双向聚类联合F": "{:.4f}",
            "分母df": "{:.0f}",
            "原始p值": "{:.6g}",
            "BH校正q值": "{:.6g}",
        }
    ).hide(axis="index")
)

display(Markdown("### Individual risk-lag coefficients (same fitted models)"))
display(
    coefficient_results.style.format(
        {
            "coefficient": "{:.8f}",
            "two_way_cluster_se": "{:.8f}",
            "t": "{:.4f}",
            "inference_df": "{:.0f}",
            "p_value": "{:.6g}",
            "effect_per_1SD_risk": "{:.8f}",
        }
    ).hide(axis="index")
)

### 主回归结果表

风险滞后p,收益滞后q,面板观测数,独立目标小时数,风险系数合计,风险+1SD的收益影响(%),95%CI下限(%),95%CI上限(%),双向聚类联合F,分子df,分母df,原始p值,BH校正q值,FDR 5%显著
1,1,1789,95,0.00012495,0.01770,-0.03011,0.06554,0.6005,1,19,0.447929,0.800289,False
2,1,1789,95,0.00016653,0.02360,-0.04062,0.08785,0.3328,2,19,0.72101,0.800289,False
3,1,1789,95,0.00008187,0.01160,-0.08178,0.10507,0.3348,3,19,0.800289,0.800289,False


### Individual risk-lag coefficients (same fitted models)

risk_lags_p,term,coefficient,two_way_cluster_se,t,inference_df,p_value,effect_per_1SD_risk
1,AI_Risk_total_lag_1,0.00012495,0.00016124,0.7749,19,0.447929,0.00017702
2,AI_Risk_total_lag_1,0.00012303,0.00016070,0.7656,19,0.453339,0.00017430
2,AI_Risk_total_lag_2,0.00004351,0.00013858,0.3140,19,0.756975,0.00006164
3,AI_Risk_total_lag_1,0.00011725,0.00016507,0.7103,19,0.486146,0.00016612
3,AI_Risk_total_lag_2,0.00003753,0.00013785,0.2723,19,0.788355,0.00005318
3,AI_Risk_total_lag_3,-0.00007291,0.00017672,-0.4126,19,0.684528,-0.00010330


## 4. Sample composition and automatic reading of the main result

The sample table makes clear which target hours survive exact lag matching. A missing early bridge window removes only the target observations that actually require it; no entire trading day is discarded automatically.

In [5]:
sample_composition = pd.concat(
    [
        sample.groupby("target_hour", observed=True).size().rename(f"p={p}")
        for p, sample in samples_by_p.items()
    ],
    axis=1,
).fillna(0).astype(int)
sample_composition.index.name = "target_window_start_NY"
display(sample_composition)

display(Markdown(f"**Observed-sample standard deviation of `AI_Risk_total`: {RISK_SD:.6g}.**"))

significant = main_results.loc[main_results["significant_BH_5pct"]]
if significant.empty:
    conclusion = (
        "After two-way clustered inference and BH correction across p=1,2,3, "
        "none of the three main specifications rejects the joint no-Granger-predictability null at 5%."
    )
else:
    orders = ", ".join(f"p={value}" for value in significant["risk_lags_p"].astype(int))
    conclusion = (
        "After two-way clustered inference and BH correction, the joint risk-lag null is rejected "
        f"at 5% for: {orders}. Inspect coefficient signs and confidence intervals before interpreting direction."
    )

display(Markdown("### Mechanical summary"))
display(Markdown(conclusion))
display(
    Markdown(
        "This is an observed-sample predictive result. It does not establish that AI-risk news "
        "causes stock-price movements, and it does not represent complete X-platform coverage."
    )
)

,p=1,p=2,p=3
target_window_start_NY,,,
10:00,109,109,109
11:00,280,280,280
12:00,320,320,320
13:00,360,360,360
14:00,360,360,360
15:00,360,360,360


**Observed-sample standard deviation of `AI_Risk_total`: 1.41677.**

### Mechanical summary

After two-way clustered inference and BH correction across p=1,2,3, none of the three main specifications rejects the joint no-Granger-predictability null at 5%.

This is an observed-sample predictive result. It does not establish that AI-risk news causes stock-price movements, and it does not represent complete X-platform coverage.

## 5. Interpretation limits

1. All 33 X News collection queries reached their result cap, so `coverage_complete` is false for every hour. `unknown_empty` and warm-up values are excluded, never changed to zero.
2. `AI_Risk_total` is the pre-registered main predictor. The other 11 indices, individual-stock regressions, symmetric `q=p` models, QQQ controls and reverse-direction tests are not run here.
3. The panel repeats the same hourly risk value across stocks. Two-way clustering prevents those repetitions from being treated as independent news shocks, but the effective time dimension remains short.
4. BIC chooses only the return-control lag order. It does not select a favourable risk result.
5. Statistical significance is not equivalent to economic importance; the one-standard-deviation effect and its confidence interval should be read alongside the p-values.